In [12]:
import os
import json
import datasets

from tqdm import tqdm

class BGEDataParser:
    def __init__(self, ds_name, root_dir = "mdata/bge-m3-data"):
        self.ds_name = ds_name
        self.root_dir = root_dir
        
    def load_all_lines(self):
        all_files = os.listdir(os.path.join(self.root_dir, self.ds_name))
        all_lines = []
        for file in all_files:
            with open(os.path.join(self.root_dir, self.ds_name, file)) as f:
                for line in f:
                    all_lines.append(json.loads(line))
            print(self.ds_name, file, len(all_lines))
        self.all_lines = all_lines
        
    def build_dataset(self):
        d_id = 0
        q_id = 0
        skip = 0
        
        corpus_id_to_text = {}
        corpus_text_to_id = {}
        queries_id_to_text = {}
        queries_text_to_id = {}
        query_pos_docs = {}
        
        for line in tqdm(self.all_lines):
            query = line["query"]
            if query in queries_text_to_id:
                skip += 1
                continue
            
            cur_q_id = str(q_id)
            q_id += 1
            queries_text_to_id[query] = cur_q_id
            queries_id_to_text[cur_q_id] = query
            query_pos_docs[cur_q_id] = []
            
            for text in line["pos"]:
                if text not in corpus_text_to_id:
                    cur_d_id = str(d_id)
                    d_id += 1
                    corpus_text_to_id[text] = cur_d_id
                    corpus_id_to_text[cur_d_id] = text
                else:
                    cur_d_id = corpus_text_to_id[text]

                query_pos_docs[cur_q_id].append(cur_d_id)
                
            for text in line["neg"]:
                if text not in corpus_text_to_id:
                    cur_d_id = str(d_id)
                    d_id += 1
                    corpus_text_to_id[text] = cur_d_id
                    corpus_id_to_text[cur_d_id] = text
                    
        self.corpus = corpus_id_to_text
        self.queries = queries_id_to_text
        self.query_pos_docs = query_pos_docs
        print(skip)
        print(len(self.corpus), len(self.queries), len(self.query_pos_docs))
        
    def save(self):
        os.makedirs(os.path.join(self.root_dir, "parsed", self.ds_name), exist_ok=True)
        with open(os.path.join(self.root_dir, "parsed", self.ds_name, "corpus.json"), "w") as f:
            json.dump(self.corpus, f)
        with open(os.path.join(self.root_dir, "parsed", self.ds_name, "queries.json"), "w") as f:
            json.dump(self.queries, f)
        with open(os.path.join(self.root_dir, "parsed", self.ds_name, "query_pos_docs.json"), "w") as f:
            json.dump(self.query_pos_docs, f)
    
parser = BGEDataParser("mMARCO-zh")
parser.load_all_lines()
parser.build_dataset()
parser.save()

mMARCO-zh mmarco_chinese_len-0-500.jsonl 100000


100%|██████████| 100000/100000 [00:02<00:00, 43268.62it/s]


0
1240099 100000 100000


In [10]:
["DuReader", "zh_NLI_data", "T2Ranking"]

['DuReader', 'zh_NLI_data', 'T2Ranking']

In [2]:
import os
import json

for dataset in ["DuReader", "zh_NLI_data", "T2Ranking"]:
    path = os.path.join("mdata/bge-m3-data/parsed",dataset)
    with open(os.path.join(path, "corpus.json"), "r") as f:
        corpus = json.load(f)
    with open(os.path.join(path, "queries.json"), "r") as f:
        queries = json.load(f)
    with open(os.path.join(path, "query_pos_docs.json"), "r") as f:
        query_pos_docs = json.load(f)
    break

In [6]:
import datasets

datasets.Dataset.load_from_disk("mdata/raw-v3/miracl/zh")[0]

{'query_id': '59350#0',
 'query': '最后一任法国国王是谁？',
 'hits': ['425353#5',
  '80135#0',
  '530277#10',
  '46587#16',
  '37537#0',
  '546263#0',
  '224533#0',
  '129022#8',
  '5539502#5',
  '129031#0',
  '4122896#0',
  '46587#32',
  '5539502#2',
  '46587#14',
  '607#42',
  '3247621#0',
  '46587#25',
  '46587#9',
  '329972#9',
  '809200#0',
  '91254#0',
  '7878177#8',
  '46587#33',
  '543#17',
  '6024534#0',
  '46587#11',
  '1013637#0',
  '46587#24',
  '84153#0',
  '1156005#2',
  '4857524#4',
  '540162#0',
  '860802#0',
  '543#15',
  '46587#7',
  '53889#0',
  '47024#0',
  '903990#0',
  '35140#0',
  '6712362#0',
  '46587#4',
  '545702#0',
  '530277#18',
  '6759186#9',
  '7274397#0',
  '46587#15',
  '530277#1',
  '46587#31',
  '530277#2',
  '7623942#3',
  '46587#0',
  '128148#1',
  '209080#0',
  '5943443#6',
  '46587#5',
  '4264552#0',
  '1596432#1',
  '46587#13',
  '6682131#0',
  '1069804#2',
  '91254#4',
  '46587#8',
  '5227545#0',
  '4078197#0',
  '322851#6',
  '607#86',
  '2458#17',
  '645

In [12]:
datasets.Dataset.load_from_disk("mdata/kd-v2-dedup/zh")[0]
datasets.Dataset.load_from_disk("data/pt-kd/wikihow")[0]

{'query': 'Be Well Read',
 'first_rank': 0,
 'docs': ['If you want to be well-read, then, in the words of William Faulkner, you\'ll have to "Read, read, read. Read everything..." You can start at the very beginning, or just make your way down an eclectic list of books that you\'d like to read.',
  'Teaching well is an art rooted in practical, applied, behavioral sciences. There are definitely techniques that have been proven to work better than the typical "stand and deliver" lecture or presenting them with only linear or sequential information such as reading or listening to lecture.',
  'Children usually learn to read beginning around the ages of 5 or 6. In the United States, this will typically be around first grade.  [1]   Though there are many methods for teaching reading to children, research suggests that teaching phonics is one of the best ways to ensure that you can help all of the children in your classroom learn to read well.',
  "A read me file (also called a readme) is a s

In [8]:
import datasets
lang="ar"
len(datasets.Dataset.load_from_disk(f"mdata/kd-v3-dedup/{lang}")),len(datasets.Dataset.load_from_disk(f"mdata/kd-v2-dedup/{lang}"))

(14852, 14852)

In [24]:
import os
import datasets

for name in os.listdir("mdata/mpt-raw"):
    ds = datasets.Dataset.load_from_disk(f"mdata/mpt-raw/{name}")
    print(name, len(ds),len(ds[10000]["docs"]))
    


T2Ranking 68385 20
DuReader 53387 20
mmarco-chinese 118936 20
zh_NLI_data 34269 20
mmarco-french 157478 20


In [23]:
from tqdm import tqdm

for name in os.listdir("mdata/mpt-raw"):
    ds = datasets.Dataset.load_from_disk(f"mdata/mpt-raw/{name}")
    ds.save_to_disk(f"mdata/mpt-raw2/{name}")
    # new_ds = {
    #     "query": [],
    #     "docs": [],
    #     "first_rank":[]
    # }
    # for i in tqdm(range(len(ds))):
    #     # if ds[i]["first_rank"]>5 or ds[i]["first_rank"]<0:
    #     #     continue
    #     new_ds["query"].append(ds[i]["query"])
    #     new_ds["docs"].append(ds[i]["docs"][:20])
    #     new_ds["first_rank"].append(ds[i]["first_rank"])
    # print(len(new_ds["query"]))
    # datasets.Dataset.from_dict(new_ds).save_to_disk(f"mdata/mpt-raw2/{name}")

Saving the dataset (3/3 shards): 100%|██████████| 157478/157478 [00:01<00:00, 136573.62 examples/s]


In [21]:
datasets.Dataset.from_dict(new_ds).save_to_disk(f"mdata/mpt-raw/{name}")

Saving the dataset (3/3 shards): 100%|██████████| 157478/157478 [00:04<00:00, 34057.13 examples/s]


In [9]:
ds2 = ds.select(range(10)).filter(lambda x: x["first_rank"] <= 10)
ds2[4]

{'query': ' 预防脚气病的维生素是',
 'docs': ['预防。吃富含硫胺素和其他维生素的适当饮食可以预防脚气病。哺乳期的母亲应该确保他们的饮食中含有所有维生素。当婴儿没有接受母乳喂养时，父母需要确保他们婴儿的婴儿配方奶粉含有硫胺素。eriberi 在美国很少见，因为现在大多数食物都富含维生素。如果你吃正常、健康的饮食，你应该摄入足够的硫胺素。今天，脚气病主要发生在酗酒的患者身上。大量饮酒会导致营养不良。',
  '脚气病是一种由维生素 B-1（硫胺素）缺乏引起的疾病。该病有两种类型：湿性脚气病和干性脚气病。湿脚气病会影响心脏功能，在最极端的情况下会影响心力衰竭。要预防脚气病，吃健康均衡的饮食，强调富含硫胺素的食物。应定期检查孕妇和哺乳期的母亲是否存在任何维生素缺乏症 还应检查婴儿配方奶粉中是否含有足够的硫胺素。任何滥用酒精的人都应该检查是否缺乏维生素 B。',
  '为了预防脚气病，请注意健康均衡的饮食，强调富含硫胺素的食物。应定期检查孕妇和哺乳期的母亲是否存在任何维生素缺乏症 还应检查婴儿配方奶粉中是否含有足够的硫胺素。任何酗酒的人都应该检查是否存在维生素 B 缺乏症。如果不加以检查和治疗，脚气病会导致死亡。然而，如果您能获得富含维生素 B-1 的食物，您患脚气病的几率就会很低。肉类、乳制品和全谷物都是维生素 B-1 的极好来源，可以防止您患上这种疾病。',
  '为了预防脚气病，请注意健康均衡的饮食，强调富含硫胺素的食物。应定期检查孕妇和哺乳期的母亲是否存在任何维生素缺乏症 还应检查婴儿配方奶粉中是否含有足够的硫胺素。任何滥用酒精的人都应该检查是否缺乏维生素 B。遗传性脚气病是一种罕见的疾病，会阻止身体吸收维生素 B-1。 2 孕妇和任何患有甲亢（甲状腺过度活跃）的人都需要额外的维生素 B-1。',
  '硫胺素——一种预防脚气病的 B 族维生素；保持食欲和生长。 aneurin，抗脚气病因子，硫胺素，维生素 B1。 B 复合物，B 族维生素，B 复合维生素，维生素 B，维生素 B 复合物，B-最初被认为是单一的维生素，但现在分成几种 B 族维生素。',
  '为了预防脚气病，请注意健康均衡的饮食，强调富含硫胺素的食物。应定期检查孕妇和哺乳期的母亲是否存在任何维生素缺乏症 还应检查婴儿配方奶粉中是否含有足够的硫胺素。任何酗酒的人都应该检查是否存在

In [11]:
client.search(index="mmarco-chinese")

{'took': 8,
 'timed_out': False,
 '_shards': {'total': 12, 'successful': 12, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 10000, 'relation': 'gte'},
  'max_score': 1.0,
  'hits': [{'_index': 'mmarco-chinese',
    '_id': '3228',
    '_score': 1.0,
    '_source': {'text': '宽阔：这条河太宽了，不能游过去。从一边到另一边测量：桌子宽三英尺。在很大程度上；大：广阔的海洋。 full：我们在光天化日之下醒来。广泛的范围或范围：现代医生必须具有广泛的医学知识。',
     'text_sparse': {':': 0.03925827145576477,
      '?': 0.0842406153678894,
      'a': 0.08244366943836212,
      'ᄄ': 0.29109421372413635,
      '不': 0.13414156436920166,
      '么': 0.40089741349220276,
      '些': 0.05786345899105072,
      '何': 0.052318159490823746,
      '光': 0.5450005531311035,
      '冰': 0.3339579105377197,
      '化': 0.4067663550376892,
      '医': 1.4246132373809814,
      '即': 0.08154398202896118,
      '味': 0.05046287178993225,
      '因': 0.11517547070980072,
      '围': 0.28670501708984375,
      '块': 0.058784693479537964,
      '处': 0.015504186972975731,
      '够': 0.04674193635582924,
 

In [41]:
import os
import json
import datasets

from tqdm import tqdm


class MmarcoDataParser:
    def __init__(self, lang, root_dir = "mdata/bge-m3-data"):
        self.lang = lang
        self.root_dir = root_dir
        
    def load_data(self):
        self.triplets_ds = datasets.load_dataset('unicamp-dl/mmarco', self.lang,trust_remote_code=True)["train"]
        self.queries_ds = datasets.load_dataset('unicamp-dl/mmarco', f'queries-{self.lang}',trust_remote_code=True)["train"]
        self.collection_ds = datasets.load_dataset('unicamp-dl/mmarco', f'collection-{self.lang}',trust_remote_code=True)["collection"]
        
    def build_dataset(self):
        corpus_id_to_text = {}
        corpus_text_to_id = {}
        queries_id_to_text = {}
        queries_text_to_id = {}
        query_pos_docs = {}
        for data in tqdm(self.collection_ds):
            _id = str(data["id"])
            text = str(data["text"])
            corpus_id_to_text[_id] = text
            corpus_text_to_id[text] = _id
        for data in tqdm(self.queries_ds):
            _id = str(data["id"])
            text = str(data["text"])
            queries_id_to_text[_id] = text
            queries_text_to_id[text] = _id
        for data in tqdm(self.triplets_ds):
            q_id = queries_text_to_id[data["query"]]
            pos_id = corpus_text_to_id[data["positive"]]
            if q_id not in query_pos_docs:
                query_pos_docs[q_id] = []
            if pos_id not in query_pos_docs[q_id]:
                query_pos_docs[q_id].append(pos_id)
            
        self.corpus = corpus_id_to_text
        self.queries = queries_id_to_text
        self.query_pos_docs = query_pos_docs
        
    def save(self):
        os.makedirs(os.path.join(self.root_dir, "parsed", "mmarco-"+self.lang), exist_ok=True)
        with open(os.path.join(self.root_dir, "parsed", "mmarco-"+self.lang, "corpus.json"), "w") as f:
            json.dump(self.corpus, f)
        with open(os.path.join(self.root_dir, "parsed", "mmarco-"+self.lang, "queries.json"), "w") as f:
            json.dump(self.queries, f)
        with open(os.path.join(self.root_dir, "parsed", "mmarco-"+self.lang, "query_pos_docs.json"), "w") as f:
            json.dump(self.query_pos_docs, f)

parser = MmarcoDataParser("chinese")
parser.load_data()
parser.build_dataset()

Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.
 70%|██████▉   | 6188486/8841823 [03:52<01:43, 25575.07it/s]

In [39]:
parser.triplets_ds[0]

{'query': '怀孕期间少量咖啡因可以吗',
 'positive': '我们不太了解怀孕期间咖啡因对您和您的宝宝的影响。因此，最好限制每天获得的数量。如果您怀孕了，请将咖啡因限制在每天 200 毫克。这大约是 1×8 盎司咖啡或一杯 12 盎司咖啡的量。',
 'negative': '孕妇吃巧克力通常是安全的，因为研究表明怀孕期间吃巧克力有某些好处。但是，孕妇应确保每天的咖啡因摄入量低于 200 毫克。'}

In [35]:
i,doc

(0,
 {'id': 0,
  'text': '科学头脑中的交流对于曼哈顿计划的成功与科学智力同等重要。笼罩着原子研究人员和工程师令人印象深刻的成就的唯一乌云是他们的成功真正意味着什么；数十万无辜的生命被抹杀。'})

In [24]:
from beir import LoggingHandler, util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/msmarco.zip"
data_path = util.download_and_unzip(url, "data/beir")

corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(
    split="train"
)

data/beir/msmarco.zip: 100%|██████████| 1.01G/1.01G [01:48<00:00, 9.95MiB/s] 
100%|██████████| 8841823/8841823 [00:48<00:00, 181734.44it/s]


In [26]:
queries

{'1185869': ')what was the immediate impact of the success of the manhattan project?',
 '1185868': '_________ justice is designed to repair the harm to victim, the community and the offender caused by the offender criminal act. question 19 options:',
 '597651': 'what color is amber urine',
 '403613': 'is autoimmune hepatitis a bile acid synthesis disorder',
 '1183785': 'elegxo meaning',
 '312651': 'how much does an average person make for tutoring',
 '80385': 'can you use a calculator on the compass test',
 '645590': 'what does physical medicine do',
 '645337': 'what does pending mean on listing',
 '186154': 'feeding rice cereal how many times per day',
 '457407': 'most dependable affordable cars',
 '441383': 'lithophile definition',
 '683408': 'what is a flail chest',
 '1164799': 'what causes ear infection in adults',
 '484187': 'put yourself on child support in texas',
 '460668': 'mushrooms health benefits',
 '666321': 'what happens in a wrist sprain',
 '182487': 'example of culture 

In [2]:
import os
import json

with open(os.path.join("mdata/bge-m3-data", "parsed", "mmarco-chinese", "corpus.json")) as f:
    corpus = json.load(f)

In [3]:
corpus

{'0': '科学头脑中的交流对于曼哈顿计划的成功与科学智力同等重要。笼罩着原子研究人员和工程师令人印象深刻的成就的唯一乌云是他们的成功真正意味着什么；数十万无辜的生命被抹杀。',
 '1': '曼哈顿计划及其原子弹帮助结束了第二次世界大战。其和平利用原子能的遗产继续对历史和科学产生影响。',
 '2': '关于曼哈顿计划的论文 - 曼哈顿计划 曼哈顿计划是为了看看是否可以制造原子弹。这个项目的成功将永远改变世界，永远让人们知道这种强大的东西可以人造。',
 '3': '曼哈顿计划是第二次世界大战期间进行的一项计划的名称，该计划旨在开发第一颗原子弹。它特指从 194 年到 1946 年在美国陆军工程兵团控制下的项目时期，由莱斯利·格罗夫斯将军管理。',
 '4': '每卷的版本以及补充网站。第一个网站“曼哈顿计划：互动历史”可在历史和遗产资源办公室网站 http://www.cfo 上找到。 doe.gov/me70/history。历史和遗产资源和国家核安全办公室',
 '5': '曼哈顿计划。这张曾经被列为机密的照片展示了第一颗原子弹——一种被原子科学家昵称为“小工具”的武器。核时代始于 1945 年 7 月 16 日，当时它在新墨西哥州的沙漠中被引爆。',
 '6': '它也不会试图取代关于原子弹和二战结束的极其丰富的文献。该收藏并不试图记录曼哈顿计划的起源和发展。',
 '7': '曼哈顿计划。曼哈顿计划是第二次世界大战期间的一项研发事业，它生产了第一批核武器。它由美国领导，得到英国和加拿大的支持。从 1942 年到 1946 年，该项目由美国陆军工程兵团的莱斯利·格罗夫斯少将指导。核物理学家罗伯特·奥本海默 (Robert Oppenheimer) 是设计实际炸弹的洛斯阿拉莫斯实验室 (Los Alamos Laboratory) 的负责人。该项目的陆军部分被指定为',
 '8': '1942 年 6 月，美国陆军工程兵团开始了曼哈顿计划——2 颗原子弹的秘密名称。',
 '9': '汉福德被选为曼哈顿项目 B 反应堆选址的主要原因之一是它靠近哥伦比亚河，哥伦比亚河是从北美海岸流入太平洋的最大河流。',
 '10': '小组讨论、社区委员会或与第三方的小组讨论，或受害者和罪犯的对话，并且需要一位熟练的协调员，他对性侵犯、家庭暴力和约会暴力以及创伤和安全问题

In [4]:
from opensearchpy import OpenSearch

client = OpenSearch("localhost:9200")

In [13]:

print(client.cat.indices())

green open nq                        bHV7MVDlS5WtHFEzqfgXNw 12 0 2681468 0     21gb     21gb
green open trec-covid                FrbUESc6SA6KJSVx8xDbpA 12 0  171331 0    2.7gb    2.7gb
green open hotpotqa                  VKUFfxCKStWJTnNJ-F3YDg 12 0 5233329 0   34.4gb   34.4gb
green open scidocs                   PNyX4s6pSkOm3xcOW_JNdA 12 0   25657 0  280.3mb  280.3mb
green open fever                     5rBkqC0qQM26_e7abM4Eng 12 0 5416568 0   38.1gb   38.1gb
green open climate-fever             -_PfbnGxSRqqcBFVqksOkQ 12 0 5416593 0     38gb     38gb
green open dbpedia-entity            h3uY9Xh8Sfekcxv6B-UHrQ 12 0 4635922 0   31.1gb   31.1gb
green open .opensearch-observability hC-g61ruR0S-X8v3ypVwfg  1 0       0 0     208b     208b
green open .plugins-ml-config        i9xceVNgTx6-BmQe3Lj1aA  1 0       1 0      4kb      4kb
green open arguana                   2Ho6wZPASwKMUUP8dDWc3A 12 0    8674 0    123mb    123mb
green open quora                     UCZw3rr3QNS837ikCZ90kg 12 0  5229

In [14]:
client.search(index="zh_nli_data")

{'took': 2,
 'timed_out': False,
 '_shards': {'total': 12, 'successful': 12, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 10000, 'relation': 'gte'},
  'max_score': 1.0,
  'hits': [{'_index': 'zh_nli_data',
    '_id': '19585',
    '_score': 1.0,
    '_source': {'text': '读《习近平的七年知青岁月》有感-中青在线',
     'text_sparse': {'"': 0.37707099318504333,
      '7': 0.7436100244522095,
      '《': 0.6989894509315491,
      '》': 0.40806588530540466,
      '七': 1.5630667209625244,
      '中': 0.8748174905776978,
      '习': 1.1337933540344238,
      '书': 0.23092569410800934,
      '些': 0.17922431230545044,
      '作': 0.7491656541824341,
      '佳': 0.02412080205976963,
      '八': 0.15943801403045654,
      '剧': 0.29255300760269165,
      '受': 0.0606246218085289,
      '句': 0.18248429894447327,
      '在': 0.28597161173820496,
      '声': 0.10993985831737518,
      '奖': 0.045809537172317505,
      '女': 0.20739519596099854,
      '好': 0.38308101892471313,
      '学': 0.4624064862728119,
      '容': 0.08